# Streaming Attention Sink：长会话滚动 KV 与能力边界

**面试问题：流式长文本怎样保留 sink、淘汰局部 token，并识别缓存已经无法回答的问题？**

## 回答主线

先明确业务目标和数据合同，再给出可比较的朴素基线；随后手写核心算法，展示中间状态、最终指标和失败路径。本 Notebook 的断言只出现在最后，用于保护关键不变量；学习重点是前面的输入、过程、对照与解释。

## 真实案例

一个售后 Agent 要处理持续到来的聊天 token，GPU 只允许每个请求保留 8 个 KV 位置。会话开头包含系统身份和“退款前必须核验订单号”的安全规则；中间包含订单号，尾部是用户追问。案例对比纯最近窗口与 sink+recent 缓存，展示每一步淘汰、规则保持和事实丢失，并在无法回答时路由到外部检索。

### 输入预览：带系统规则和订单事实的长会话

In [1]:
stream = ["<SYS>", "售后助手", "规则:先核验订单号", "用户", "订单", "A-7842", "已签收", "商品破损", "请求退款", "客服", "请上传照片", "用户", "照片已上传", "现在能退款吗"]  # 构造十四个具有真实客服语义的流式 token。
capacity = 8  # 设定教学 GPU 只能保留八个 KV 位置。
sink_size = 3  # 固定保留系统标记、身份和高优先级规则三个 sink token。
print(f"完整会话长度={len(stream)}，KV 容量={capacity}，sink={sink_size}")  # 展示长会话和有限缓存之间的矛盾。
for index, token in enumerate(stream):  # 逐项展示绝对位置和可读 token。
    print(f"position={index:02d} token={token}")  # 强调滚动缓存仍需保存原始绝对位置。

完整会话长度=14，KV 容量=8，sink=3
position=00 token=<SYS>
position=01 token=售后助手
position=02 token=规则:先核验订单号
position=03 token=用户
position=04 token=订单
position=05 token=A-7842
position=06 token=已签收
position=07 token=商品破损
position=08 token=请求退款
position=09 token=客服
position=10 token=请上传照片
position=11 token=用户
position=12 token=照片已上传
position=13 token=现在能退款吗


## Baseline 基线：只保留最近八个 token

In [2]:
def recent_window(tokens, limit):  # 实现最朴素的固定长度滑动窗口。
    start = max(0, len(tokens) - limit)  # 计算最近窗口在原序列中的起始绝对位置。
    return list(enumerate(tokens[start:], start=start))  # 返回绝对位置与 token，避免位置重编号。

baseline_cache = recent_window(stream, capacity)  # 对完整会话应用纯最近窗口策略。
baseline_tokens = [token for _, token in baseline_cache]  # 提取缓存中的可读 token 便于规则检查。
print("纯 recent window 最终缓存：", baseline_cache)  # 展示基线保留的具体位置和内容。
print("系统规则仍在缓存吗？", "规则:先核验订单号" in baseline_tokens)  # 暴露纯窗口已经淘汰高优先级规则。

纯 recent window 最终缓存： [(6, '已签收'), (7, '商品破损'), (8, '请求退款'), (9, '客服'), (10, '请上传照片'), (11, '用户'), (12, '照片已上传'), (13, '现在能退款吗')]
系统规则仍在缓存吗？ False


### 核心实现：sink 与 recent 分区保留

In [3]:
def sink_recent_cache(tokens, limit, fixed_sink):  # 实现固定 sink 加最近窗口的逻辑缓存。
    sink = list(enumerate(tokens[:fixed_sink]))  # 永久保留最前面的系统与规则位置。
    recent_budget = max(0, limit - len(sink))  # 计算扣除 sink 后可供近期 token 使用的容量。
    recent_start = max(fixed_sink, len(tokens) - recent_budget)  # 计算 recent 分区的起始绝对位置。
    recent = list(enumerate(tokens[recent_start:], start=recent_start))  # 保留尾部近期交互并维持原始位置。
    return sink + recent  # 按因果顺序合并 sink 与 recent 两个分区。

timeline = []  # 收集每个到达步骤的缓存快照用于教学观察。
for end in range(1, len(stream) + 1):  # 模拟 token 逐个进入 decode 服务。
    snapshot = sink_recent_cache(stream[:end], capacity, sink_size)  # 在每个时间点重新计算逻辑缓存。
    timeline.append(snapshot)  # 保存快照供后续分析和回放。
print("关键时间点的 sink+recent 缓存：")  # 输出状态迁移标题。
for step in (3, 6, 9, 12, 14):  # 选择能体现增长和淘汰的五个时间点。
    print(f"step={step:02d} cache={timeline[step - 1]}")  # 展示 sink 不动而 recent 分区滚动。

关键时间点的 sink+recent 缓存：
step=03 cache=[(0, '<SYS>'), (1, '售后助手'), (2, '规则:先核验订单号')]
step=06 cache=[(0, '<SYS>'), (1, '售后助手'), (2, '规则:先核验订单号'), (3, '用户'), (4, '订单'), (5, 'A-7842')]
step=09 cache=[(0, '<SYS>'), (1, '售后助手'), (2, '规则:先核验订单号'), (4, '订单'), (5, 'A-7842'), (6, '已签收'), (7, '商品破损'), (8, '请求退款')]
step=12 cache=[(0, '<SYS>'), (1, '售后助手'), (2, '规则:先核验订单号'), (7, '商品破损'), (8, '请求退款'), (9, '客服'), (10, '请上传照片'), (11, '用户')]
step=14 cache=[(0, '<SYS>'), (1, '售后助手'), (2, '规则:先核验订单号'), (9, '客服'), (10, '请上传照片'), (11, '用户'), (12, '照片已上传'), (13, '现在能退款吗')]


## 结果解读：规则保持与事实可用性是两件事

In [4]:
final_cache = timeline[-1]  # 读取完整会话结束时的 sink+recent 缓存。
final_tokens = [token for _, token in final_cache]  # 提取最终缓存的 token 内容。
policy_present = "规则:先核验订单号" in final_tokens  # 检查系统安全规则是否仍可见。
order_present = "A-7842" in final_tokens  # 检查中段订单号事实是否仍可见。
recent_request_present = "现在能退款吗" in final_tokens  # 检查最新用户意图是否仍可见。
print("最终 sink+recent 缓存：", final_cache)  # 展示最终实际保留的绝对位置。
print(f"规则保留={policy_present}，订单事实保留={order_present}，最新请求保留={recent_request_present}")  # 同时展示三类上下文的不同命运。
print("解读：sink 解决注意力稳定和关键前缀保留，但不会神奇恢复被 recent 窗口淘汰的中段事实。")  # 明确 StreamingLLM 类方法的能力边界。

最终 sink+recent 缓存： [(0, '<SYS>'), (1, '售后助手'), (2, '规则:先核验订单号'), (9, '客服'), (10, '请上传照片'), (11, '用户'), (12, '照片已上传'), (13, '现在能退款吗')]
规则保留=True，订单事实保留=False，最新请求保留=True
解读：sink 解决注意力稳定和关键前缀保留，但不会神奇恢复被 recent 窗口淘汰的中段事实。


## 失败案例：模型被要求回答已淘汰的订单号

In [5]:
def answer_order_question(cache):  # 实现带可验证拒答的订单号查询函数。
    cached_tokens = [token for _, token in cache]  # 提取缓存 token 供订单号模式检查。
    order_ids = [token for token in cached_tokens if token.startswith("A-")]  # 查找仍驻留缓存的订单号。
    if not order_ids:  # 缓存中没有权威订单事实时不能编造答案。
        return {"status": "NEED_RETRIEVAL", "answer": None, "reason": "订单号已被滚动缓存淘汰"}  # 返回明确的外部检索路由。
    return {"status": "ANSWER", "answer": order_ids[-1], "reason": "订单号仍在当前 KV"}  # 仅在事实存在时回答。

failure_result = answer_order_question(final_cache)  # 尝试从最终有限缓存回答订单号问题。
retrieved_cache = final_cache + [(5, "A-7842")]  # 模拟从权威会话日志检索回原始位置五的订单号。
recovered_result = answer_order_question(retrieved_cache)  # 在补回证据后重新回答。
print("直接回答路径：", failure_result)  # 展示有限窗口下必须拒答的失败结果。
print("检索补证后：", recovered_result)  # 展示外部记忆如何恢复事实而不篡改 sink 语义。

直接回答路径： {'status': 'NEED_RETRIEVAL', 'answer': None, 'reason': '订单号已被滚动缓存淘汰'}
检索补证后： {'status': 'ANSWER', 'answer': 'A-7842', 'reason': '订单号仍在当前 KV'}


### 生产边界

In [6]:
cache_manifest = {"request_id": "chat-20260729-17", "capacity": capacity, "sink_positions": [0, 1, 2], "recent_positions": [position for position, _ in final_cache[sink_size:]], "rope_coordinate": "absolute", "model_revision": "support-7b-r3"}  # 构造防止错误 KV 复用的版本化缓存清单。
print("缓存 manifest：", cache_manifest)  # 展示容量、位置坐标和模型版本等关键合同。
print("生产替换点：真实系统还需 block manager、RoPE/位置版本、prefill 重算、跨请求隔离、日志检索和问题可回答性分类器。")  # 明确列表缓存与 GPU KV 服务的差距。

缓存 manifest： {'request_id': 'chat-20260729-17', 'capacity': 8, 'sink_positions': [0, 1, 2], 'recent_positions': [9, 10, 11, 12, 13], 'rope_coordinate': 'absolute', 'model_revision': 'support-7b-r3'}
生产替换点：真实系统还需 block manager、RoPE/位置版本、prefill 重算、跨请求隔离、日志检索和问题可回答性分类器。


## 回归测试：只保护容量、位置和拒答

In [7]:
assert len(final_cache) == capacity  # 验证 sink 与 recent 合计没有突破 KV 容量。
assert [position for position, _ in final_cache[:sink_size]] == [0, 1, 2]  # 验证三个 sink 位置始终保持原始绝对坐标。
assert policy_present and recent_request_present  # 验证系统规则和最新用户请求同时存在。
assert not order_present  # 验证失败案例中的中段订单号确实已被淘汰。
assert failure_result["status"] == "NEED_RETRIEVAL"  # 验证缺失事实时系统选择检索而不是编造。
assert recovered_result["answer"] == "A-7842"  # 验证权威补证后能够恢复正确答案。
print("回归测试通过：容量、绝对位置、规则保持、事实淘汰与检索恢复均符合合同。")  # 用可见消息总结测试覆盖。

回归测试通过：容量、绝对位置、规则保持、事实淘汰与检索恢复均符合合同。
